# STAC job outputs with Cuiman

Start the local testing service in a separate terminal with `pixi run serve`, then run this notebook from the repository root in the Pixi environment. The service listens at `http://127.0.0.1:8008` by default. Set `EOZILLA_TESTING_STAC_DIR` **before** starting it if you want an isolated artifact directory.

This demo submits both `create_inline_stac` and `create_linked_stac`, inspects original results and native PySTAC objects, applies a branded-client transformation, and reads a CSV Asset. The service keeps generated files until you remove its artifact directory.

In [ ]:
import pandas as pd
import pystac
from cuiman import Client
from examples.guides.cuiman.stac import DemoClientConfig
from gavicore.models import ProcessRequest

`DemoClientConfig` registers a composed STAC opener. Its async transform marks loaded Items with `demo:reviewed`; it does not alter the raw OGC result or follow remote STAC links. The source is [`stac.py`](https://github.com/eo-tools/eozilla/blob/main/examples/guides/cuiman/stac.py).

In [ ]:
client = Client(
    api_url="http://127.0.0.1:8008",
    auth={"auth_type": "none"},
    config_type=DemoClientConfig,
)

## Submit the two processes

Each execution creates a new job and a separate run directory on the service. Re-running this cell creates more jobs.

In [ ]:
job_ids = {}
for process_id in ("create_inline_stac", "create_linked_stac"):
    job = client.execute_process(
        process_id, request=ProcessRequest(inputs={"item_count": 2})
    )
    job_ids[process_id] = job.jobID
job_ids

## Inspect raw and native results

Opening the selected `item` waits for job completion. The linked process fetches its selected STAC document over HTTP; inline metadata needs no metadata fetch. `get_job_results()` still returns the original values and all output names.

In [ ]:
opened = {}
for process_id, job_id in job_ids.items():
    item = client.open_job_result(
        job_id, output_name="item", data_type=pystac.Item
    )
    collection = client.open_job_result(
        job_id, output_name="item_collection", data_type=pystac.ItemCollection
    )
    raw = client.get_job_results(job_id)
    assert raw.root is not None
    opened[process_id] = {"item": item, "collection": collection, "raw": raw}
    print(process_id, sorted(raw.root), [member.id for member in collection])
    print("raw item type:", type(raw.root["item"]).__name__)
    print("native item:", item.id, sorted(item.assets))

## Check the transformation and open one Asset

The mark appears only on returned native Items. Asset opening reads the exact selected CSV Asset through Cuiman's pandas reader and does not rerun the STAC transform. The `data` Asset points to Zarr; HTTP Zarr reading needs optional `fsspec[http]` dependencies not installed in the default Pixi environment.

In [ ]:
reports = {}
for process_id, value in opened.items():
    item = value["item"]
    collection = value["collection"]
    assert item.properties["demo:reviewed"] is True
    assert all(member.properties["demo:reviewed"] for member in collection)
    report_asset = item.assets["report"]
    reports[process_id] = client.open_job_result(
        report_asset, data_type=pd.DataFrame
    )
    print(process_id, report_asset.href, reports[process_id]["mean_ndvi"].tolist())
assert reports["create_inline_stac"].equals(reports["create_linked_stac"])

## Cleanup

Close the client. Then stop the testing service and remove only the generated run directories you no longer need from `EOZILLA_TESTING_STAC_DIR` (default `.pixi/testing-stac`). The service has no API for deleting those files. Following native PySTAC links later uses PySTAC navigation and does not run the composition transform on newly fetched documents.

In [ ]:
client.close()
print("Closed client. Remove the testing service's generated run directories when finished.")